# DipRadar v2 — Training Notebook

**Objectivo:** Treinar e avaliar os modelos XGBoost de regressão (upside + downside) 
comparando com os modelos de produção v1.

**Fluxo:**
1. Setup (instalar deps, clonar repo, upload alert_db)
2. Build dataset (fetch preços + targets + features v2)
3. Train/val split temporal
4. Treinar v2 (2× XGBoost)
5. Avaliar (Spearman, Top-K, Profit Sim)
6. Feature importance
7. Guardar modelos (download .pkl)

> ⚠️ **Anti-leakage:** features e targets são calculados em janelas estritamente separadas.

## 1. Setup

In [ ]:
# Instalar dependências
!pip install -q xgboost yfinance scipy pyarrow

In [ ]:
import os

# Clonar o repo (substitui pelo teu token se o repo for privado)
GITHUB_TOKEN = ''  # @param {type:'string'}

if GITHUB_TOKEN:
    !git clone https://{GITHUB_TOKEN}@github.com/romeurf/DipRadar.git
else:
    !git clone https://github.com/romeurf/DipRadar.git

os.chdir('DipRadar')
!pwd && ls

In [ ]:
from google.colab import files
import shutil, os

print('Upload o teu alert_db.csv aqui:')
uploaded = files.upload()

# Mover para /data/ como o label_resolver espera
os.makedirs('/data', exist_ok=True)
for fname in uploaded:
    shutil.copy(fname, f'/data/{fname}')
    print(f'Copiado para /data/{fname}')

# Verificar
import pandas as pd
db = pd.read_csv('/data/alert_db.csv')
print(f'Linhas: {len(db)} | Colunas: {list(db.columns)}')
db.head(3)

## 2. Build Dataset

In [ ]:
import sys
sys.path.insert(0, '.')

from experiments.ml_v2.build_dataset import load_alert_db, fetch_all_prices, build_dataset
from pathlib import Path
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# 1. Carregar alertas resolvidos
df_alerts = load_alert_db(Path('/data/alert_db.csv'), min_samples=10)
print(f'Alertas resolvidos: {len(df_alerts)}')
print(f"Win rate v1: {df_alerts['label_win'].mean():.1%}")
df_alerts[['ticker', 'alert_date', 'entry_price', 'label_win']].head(5)

In [ ]:
# 2. Fetch de preços (1 request por ticker)
# Pode demorar alguns minutos se houver muitos tickers
price_cache = fetch_all_prices(df_alerts)
print(f'Tickers com preços: {len(price_cache)}')

In [ ]:
# 3. Construir dataset com features v2 + targets sequence-aware
df_ds = build_dataset(df_alerts, price_cache)
print(f'Shape: {df_ds.shape}')
print()
print('Distribuição dos targets:')
print(df_ds[['max_return_60d', 'max_drawdown_60d']].describe().round(3))
df_ds.head(3)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df_ds['max_return_60d'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('max_return_60d (upside)')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.7)

axes[1].hist(df_ds['max_drawdown_60d'], bins=30, color='tomato', edgecolor='white')
axes[1].set_title('max_drawdown_60d (downside)')
axes[1].axvline(0, color='red', linestyle='--', alpha=0.7)

# Log transform comparison
y_up_t = np.log1p(df_ds['max_return_60d'].clip(lower=0))
axes[2].hist(y_up_t, bins=30, color='mediumseagreen', edgecolor='white')
axes[2].set_title('log1p(max_return_60d) — após transform')

plt.tight_layout()
plt.show()
print('Nota: log transform torna a distribuição mais simétrica.')

## 3. Train / Val Split (temporal)

In [ ]:
from experiments.ml_v2.pipeline import FEATURE_COLUMNS_V2

# Split temporal: 80% mais antigos para treino, 20% mais recentes para validação
# NUNCA fazer split aleatório — leakage temporal!
df_ds = df_ds.sort_values('alert_date').reset_index(drop=True)

split_idx = int(len(df_ds) * 0.80)
df_train  = df_ds.iloc[:split_idx]
df_val    = df_ds.iloc[split_idx:]

print(f'Train: {len(df_train)} amostras ({df_train["alert_date"].min().date()} → {df_train["alert_date"].max().date()})')
print(f'Val  : {len(df_val)}   amostras ({df_val["alert_date"].min().date()} → {df_val["alert_date"].max().date()})')

# Feature matrix e targets
feat_cols = [c for c in FEATURE_COLUMNS_V2 if c in df_ds.columns]

X_train    = df_train[feat_cols]
y_up_train = df_train['max_return_60d'].values
y_dn_train = df_train['max_drawdown_60d'].values

X_val      = df_val[feat_cols]
y_up_val   = df_val['max_return_60d'].values
y_dn_val   = df_val['max_drawdown_60d'].values

print(f'Features: {len(feat_cols)} — {feat_cols}')

## 4. Treinar Modelos v2

In [ ]:
from experiments.ml_v2.pipeline import train_v2

models_v2 = train_v2(X_train, y_up_train, y_dn_train, feature_cols=feat_cols)

print(f'Treino concluído.')
print(f'  Amostras : {models_v2.n_train_samples}')
print(f'  Data     : {models_v2.train_date}')
print(f'  Features : {len(models_v2.feature_cols)}')

## 5. Avaliar

In [ ]:
from experiments.ml_v2.pipeline import predict_v2
from experiments.ml_v2.evaluation import evaluate_v2, print_report

# Predições no validation set
entry_prices_val = df_val['entry_price'].values
pred_df = predict_v2(X_val, models_v2, entry_prices=entry_prices_val)

print(f'Predições geradas: {len(pred_df)}')
print(f'Rejeitados por filtros: {pred_df["rejected"].sum()}')
pred_df.head(5)

In [ ]:
report = evaluate_v2(pred_df, y_up_val, y_dn_val)
print_report(report)

print('Interpretar:')
print(f"  Spearman upside   > 0.25 ? {'✅' if report['spearman_up'] > 0.25 else '❌'} ({report['spearman_up']:.4f})")
print(f"  Top-10% precision > random baseline ? ", end='')
top10 = report['top_k_precision']['top_10pct']['precision']
base  = report['top_10pct_baseline_random']['precision']
print(f"{'✅' if top10 > base else '❌'} ({top10:.2%} vs {base:.2%} random)")

## 6. Feature Importance

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

def plot_importance(model, feature_cols, title, ax, top_n=15):
    imp = pd.Series(model.feature_importances_, index=feature_cols)
    imp = imp.sort_values(ascending=True).tail(top_n)
    imp.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('Importance')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plot_importance(models_v2.model_up,   feat_cols, 'Upside model (max_return_60d)',   axes[0])
plot_importance(models_v2.model_down, feat_cols, 'Downside model (max_drawdown_60d)', axes[1])
plt.tight_layout()
plt.savefig('feature_importance_v2.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: feature_importance_v2.png')

In [ ]:
# Scatter: predicted vs real (upside)
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (pred_col, real_arr, title, color) in zip(axes, [
    ('pred_up',   y_up_val,  'Upside',   'steelblue'),
    ('pred_down', y_dn_val, 'Downside', 'tomato'),
]):
    pred = pred_df[pred_col].values
    corr, _ = spearmanr(pred, real_arr)
    ax.scatter(pred, real_arr, alpha=0.5, s=20, color=color)
    ax.set_xlabel(f'Predicted {title}')
    ax.set_ylabel(f'Real {title}')
    ax.set_title(f'{title} | Spearman = {corr:.3f}')
    ax.axline((0,0), slope=1, color='black', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('predicted_vs_real_v2.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Guardar Modelos

In [ ]:
from experiments.ml_v2.pipeline import save_models_v2
from pathlib import Path
import json

# Verificar critérios mínimos antes de guardar
spearman_ok = report['spearman_up'] > 0.25
top10_ok    = report['top_k_precision']['top_10pct']['precision'] > report['top_10pct_baseline_random']['precision']

print('Critérios de promoção:')
print(f"  Spearman > 0.25  : {'✅ PASS' if spearman_ok else '❌ FAIL'}")
print(f"  Top-10% > random : {'✅ PASS' if top10_ok else '❌ FAIL'}")
print()

save_models_v2(models_v2, path=Path('experiments/ml_v2'))

# Guardar report
with open('experiments/ml_v2/eval_report_v2.json', 'w') as f:
    def _serialize(o):
        if isinstance(o, (int, float, str, bool)): return o
        if isinstance(o, dict): return {k: _serialize(v) for k, v in o.items()}
        return str(o)
    json.dump(_serialize(report), f, indent=2)

print('Guardado: dip_models_v2.pkl + eval_report_v2.json')
print()

# Download
from google.colab import files
files.download('experiments/ml_v2/dip_models_v2.pkl')
files.download('experiments/ml_v2/eval_report_v2.json')

## ✅ Próximos Passos

Se os critérios forem satisfeitos:

1. **Fazer download** do `dip_models_v2.pkl` e `eval_report_v2.json`
2. **Commit** dos ficheiros para `experiments/ml_v2/`
3. Atualizar `ml_predictor.py` para usar `load_models_v2()` em paralelo com v1
4. Comparar outputs lado-a-lado em produção por 2–4 semanas
5. Só então substituir `dip_model_stage1.pkl` / `dip_model_stage2.pkl`

**Não substituir em produção directamente** — shadow mode primeiro.